# Single agent, ensemble, and hierarchy without thinking

`find_failures.ipynb` showed that with Qwen3 thinking mode off, the single agent (Qwen3-4B) gets about 47% of the development problems right, fails by wrong arithmetic rather than by running out of tokens, and uses about 2.7k tokens whatever its budget. It got about 70% at depth 2 and 25% at depth 6.

This notebook runs all three systems on the same 40 problems with thinking off for every call:

- **Single agent:** one chat with the whole budget.
- **Ensemble:** three independent chats with a third of the budget each, scored by 2-of-3 majority vote.
- **Hierarchy:** a planner, two workers, and a merge step, each released a quarter of the budget, with unused tokens carried forward.

Budgets are 12k and 16k, so each ensemble attempt (4k or 5.3k) and hierarchy stage (3k or 4k) has room for the roughly 3k tokens one solve needs. Failures here should come from reasoning, not from starved budgets.

Run it as natalia@siwek.poznan.pl, whose Drive holds the `find_failures` results, with **Runtime > Change runtime type > L4 GPU**. Results go to `MyDrive/small-multi-agents/results/dev_systems/`, with the problems, a `run.log`, and summary CSV files. If the session disconnects, rerun every cell. Runs resume from the last saved batch.

In [1]:
RUN_NAME = "dev_systems"
PROBLEMS_FROM = "find_failures/no_thinking"   # reuse that option's problems.jsonl
SYSTEMS = ["single_agent", "ensemble", "hierarchy"]
BUDGETS = [12000, 16000]
SAMPLING_SEED = 42
FLAGS = ["--no-thinking"]

In [2]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
NVIDIA L4, 23034 MiB


In [3]:
from google.colab import drive

drive.mount("/content/drive")
RESULTS = "/content/drive/MyDrive/small-multi-agents/results"
OUT = f"{RESULTS}/{RUN_NAME}"
!mkdir -p {OUT}

Mounted at /content/drive


In [4]:
REPO = "/content/small-multi-agents"
!test -d {REPO} || git clone https://github.com/liatzheng/small-multi-agents.git {REPO}
%cd {REPO}
!git pull --ff-only && git log -1 --format='%H %s'

Cloning into '/content/small-multi-agents'...
remote: Enumerating objects: 164, done.
remote: Counting objects: 100% (164/164), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 164 (delta 95), reused 98 (delta 38), pack-reused 0 (from 0)
Receiving objects: 100% (164/164), 141.00 KiB | 14.10 MiB/s, done.
Resolving deltas: 100% (95/95), done.
/content/small-multi-agents
Already up to date.
1c5b1a0c9fa2a013188f9db8a2dbba80baa1fd74 Add a notebook running all three systems without thinking


Hugging Face downloads use your token if it is stored as a Colab secret: open the key icon in the left sidebar, add a secret named `HF_TOKEN`, and turn on notebook access. The token stays in your Colab account, never in the notebook or the repo.

In [5]:
from notebook_tools import use_hf_token

use_hf_token()

Using the HF_TOKEN Colab secret.


vLLM 0.33.0 brings its own matching torch, torchvision, and transformers, replacing Colab's. The runner runs as a subprocess, so no runtime restart is needed.

In [6]:
!pip install -q -r requirements-vllm.txt
!python -c "import vllm, torch, transformers; print('vllm', vllm.__version__, 'torch', torch.__version__, 'transformers', transformers.__version__, torch.cuda.get_device_name())" | tee {OUT}/environment.txt
!git rev-parse HEAD | tee -a {OUT}/environment.txt

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
numba-cuda 0.28.2 requires cuda-core<1.0.0,>=0.5.1, but you have cuda-core 1.3.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
cuml-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
cudf-cu13 26.6.0 requires numba<0.65.0,>=0.60.0, but you have numba 0.65.0 which is incompatible.
vllm 0.33.0 torch 2.14.0+cu130 transformers 5.20.2 NVIDIA L4
1c5b1a0c9fa2a013188f9db8a2dbba80

## Problems

A copy of the `find_failures` problems, so this folder holds everything its runs used.

In [7]:
import shutil

PROBLEMS = f"{OUT}/problems.jsonl"
shutil.copyfile(f"{RESULTS}/{PROBLEMS_FROM}/problems.jsonl", PROBLEMS)
!wc -l {PROBLEMS}

40 /content/drive/MyDrive/small-multi-agents/results/dev_systems/problems.jsonl


## Run

Every system at every budget, loading the model once. Everything the runner prints is also appended to `run.log`.

In [8]:
from notebook_tools import run_logged

run_logged(["python", "run_vllm.py", PROBLEMS, "--out-dir", OUT, "--systems", *SYSTEMS,
            "--budgets", *map(str, BUDGETS), "--seed", str(SAMPLING_SEED), *FLAGS, "--resume"],
           f"{OUT}/run.log")

=== python run_vllm.py /content/drive/MyDrive/small-multi-agents/results/dev_systems/problems.jsonl --out-dir /content/drive/MyDrive/small-multi-agents/results/dev_systems --systems single_agent ensemble hierarchy --budgets 12000 16000 --seed 42 --no-thinking --resume
=== started 2026-11-22T19:44:19+00:00
W1122 19:44:35.007000 2069 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1122 19:44:35.008000 2069 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1122 19:44:35.008000 2069 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead
(MainProcess pid=2069) INFO 11-22 19:44:40 [startup.py:96] non-default args: {'dtype': 'bfloat16', 'max_model_len': 16000, 'disable_log_stats': True, 'revision': '1cfa9a7208912126459214e8b04321603b3df60c', 'tokenizer_revision': '1cfa9a7208912126459214e8b04321603b3df60c', 'model': 'Qwen/Qwen3-4B'}
(MainProcess pid=206

## Results

Accuracy by system and budget. `length_limit` counts problems where the model ran out of tokens (for the ensemble, any attempt; for the hierarchy, any stage). `no_majority` is the share of ensemble problems where no two attempts agreed, and `protocol_failure` the share of hierarchy problems that stopped early for another reason: an unreadable plan or an empty report.

In [9]:
import json
from pathlib import Path

import pandas as pd
from notebook_tools import load_runs

runs = {path: [json.loads(line) for line in path.read_text().splitlines()]
        for path in sorted(Path(OUT).glob("b*/*.jsonl"))}
df = pd.DataFrame(load_runs(OUT))


def summarize(df, by):
    return df.groupby(by, sort=False).agg(
        accuracy=("correct", "mean"),
        problems=("correct", "size"),
        wrong_answer=("status", lambda s: (s == "wrong_answer").mean()),
        length_limit=("ran_out", "mean"),
        no_majority=("status", lambda s: (s == "no_majority").mean()),
        protocol_failure=("status", lambda s: s.isin(["invalid_plan", "empty_response"]).mean()),
        mean_total=("total", "mean"),
    ).round(2)


df.to_csv(f"{OUT}/summary_problems.csv", index=False)
by_system = summarize(df, ["system", "budget"])
by_system.to_csv(f"{OUT}/summary_by_system.csv")
by_system

,,accuracy,problems,wrong_answer,length_limit,no_majority,protocol_failure,mean_total
system,budget,,,,,,,
ensemble,12000,0.40,40,0.22,0.02,0.38,0.0,8258.95
hierarchy,12000,0.15,40,0.72,0.08,0.00,0.0,4631.27
single_agent,12000,0.42,40,0.50,0.02,0.00,0.0,3137.95
ensemble,16000,0.40,40,0.20,0.02,0.40,0.0,8822.88
hierarchy,16000,0.25,40,0.65,0.05,0.00,0.0,4531.40
single_agent,16000,0.48,40,0.45,0.02,0.00,0.0,3164.75


Accuracy by depth and problem kind: rows are system and budget, columns depth and kind.

In [10]:
by_depth = summarize(df, ["system", "budget", "d", "kind"])
by_depth.to_csv(f"{OUT}/summary_by_depth.csv")
by_depth["accuracy"].unstack(["d", "kind"])

d                             2                     6          
kind                independent dependent independent dependent
system       budget                                            
ensemble     12000          0.6       0.7         0.0       0.3
             16000          0.7       0.6         0.2       0.1
hierarchy    12000          0.3       0.3         0.0       0.0
             16000          0.4       0.4         0.1       0.1
single_agent 12000          0.7       0.6         0.2       0.2
             16000          0.8       0.6         0.1       0.4

## Ensemble attempts

`attempt_accuracy` is the share of single attempts that were right, `any_correct` the share of problems where at least one attempt was right, and `vote_accuracy` the share the majority vote got right. Voting can only help where attempts are right more often than not and their mistakes differ.

In [11]:
rows = []
for path, records in runs.items():
    if path.stem != "ensemble":
        continue
    for r in records:
        statuses = [a["status"] for a in r["attempts"]]
        rows.append({"budget": r["budget"], "d": r["chain_length"], "kind": r["kind"],
                     "attempt_accuracy": statuses.count("correct") / 3,
                     "any_correct": "correct" in statuses, "vote_accuracy": r["correct"],
                     "attempt_output": sum(a["output_tokens"] for a in r["attempts"]) / 3})
if rows:
    ensemble = pd.DataFrame(rows).groupby(["budget", "d", "kind"]).mean().round(2)
    ensemble.to_csv(f"{OUT}/summary_ensemble.csv")
    display(ensemble)
else:
    print("No ensemble runs yet.")

attempt_accuracy  any_correct  vote_accuracy  \
budget d kind                                                        
12000  2 dependent                0.63          0.8            0.7   
         independent              0.60          0.9            0.6   
       6 dependent                0.27          0.4            0.3   
         independent              0.03          0.1            0.0   
16000  2 dependent                0.63          0.9            0.6   
         independent              0.67          0.8            0.7   
       6 dependent                0.17          0.4            0.1   
         independent              0.13          0.2            0.2   

                      attempt_output  
budget d kind                         
12000  2 dependent           2186.00  
         independent         1886.60  
       6 dependent           2743.77  
         independent         2553.17  
16000  2 dependent           2562.53  
         independent         2192.93  
       6 dependent           2767.53  
         independent         2598.43

## Hierarchy stages

How each stage spends its tokens, where unsolved problems stopped, and how often the planner asked worker 2 to use worker 1's result. Dependent problems need that hand-off; independent ones do not.

In [12]:
import re

calls, plans = [], []
for path, records in runs.items():
    if path.stem != "hierarchy":
        continue
    for r in records:
        for call in r["calls"]:
            calls.append({"budget": r["budget"], "stage": call["stage"], "cap": call["budget"],
                          "output": call["output_tokens"], "stop_reason": call["stop_reason"]})
        found = re.search(r'"pass_first"\s*:\s*(true|false)', r["calls"][0]["final_text"]) if r["calls"] else None
        plans.append({"budget": r["budget"], "kind": r["kind"],
                      "pass_first": found.group(1) if found else "unreadable"})
if calls:
    stages = pd.DataFrame(calls).groupby(["budget", "stage"], sort=False).agg(
        calls=("output", "size"),
        mean_cap=("cap", "mean"),
        mean_output=("output", "mean"),
        max_output=("output", "max"),
        ran_out=("stop_reason", lambda s: (s != "stop").mean()),
    ).round(2)
    stages.to_csv(f"{OUT}/summary_hierarchy_stages.csv")
    display(stages)
    failed = df[(df.system == "hierarchy") & ~df.correct].groupby(
        ["budget", "failed_stage", "status"], dropna=False).size().rename("problems")
    display(failed)
    display(pd.DataFrame(plans).groupby(["budget", "kind"]).pass_first.value_counts().unstack(fill_value=0))
else:
    print("No hierarchy runs yet.")

calls  mean_cap  mean_output  max_output  ran_out
budget stage                                                      
12000  planner      40   3000.00        65.28          84     0.00
       worker_1     40   5466.12       848.72        5015     0.05
       worker_2     38   7380.58        61.97        1275     0.00
       merge        38   9232.47       612.61        9279     0.03
16000  planner      40   4000.00        64.80         120     0.00
       worker_1     40   7466.60       923.55        7015     0.05
       worker_2     38  10409.58        75.13        1275     0.00
       merge        38  13273.29       456.89        2670     0.00

budget  failed_stage  status      
12000   merge         format_error     2
                      length_limit     1
        worker_1      length_limit     2
        NaN           wrong_answer    29
16000   merge         format_error     2
        worker_1      length_limit     2
        NaN           wrong_answer    26
Name: problems, dtype: int64

pass_first          true
budget kind             
12000  dependent      20
       independent    20
16000  dependent      20
       independent    20

## Compared with `find_failures`

The single agent here uses the same problems, model, sampling seed, and thinking setting as the `no_thinking` option there, so at 16k it should give mostly the same answers. vLLM batches differ between runs, so a few can change.

In [13]:
before = pd.DataFrame(load_runs(f"{RESULTS}/{PROBLEMS_FROM}")).assign(run=PROBLEMS_FROM)
now = df[df.system == "single_agent"].assign(run=RUN_NAME)
both = pd.concat([before, now])
summarize(both, ["run", "budget"])

accuracy  problems  wrong_answer  \
run                       budget                                     
find_failures/no_thinking 16000       0.48        40          0.50   
                          8000        0.48        40          0.48   
dev_systems               12000       0.42        40          0.50   
                          16000       0.48        40          0.45   

                                  length_limit  no_majority  protocol_failure  \
run                       budget                                                
find_failures/no_thinking 16000           0.02          0.0               0.0   
                          8000            0.02          0.0               0.0   
dev_systems               12000           0.02          0.0               0.0   
                          16000           0.02          0.0               0.0   

                                  mean_total  
run                       budget              
find_failures/no_thinking 16000      3127.65  
                          8000       3036.22  
dev_systems               12000      3137.95  
                          16000      3164.75

## Reading the results

- The hierarchy beating both the single agent and the ensemble at the same budget, most at depth 6 or on dependent problems, is the pattern the proposal predicts.
- The ensemble should help most where single attempts are right more than half the time (depth 2) and little where they are mostly wrong (depth 6).
- Many `protocol_failure` or `length_limit` results for the hierarchy point to a prompt or budget problem in one stage. The stage table shows which.
- With 10 pairs per depth and kind, each problem is 10 percentage points within a cell, so treat gaps under about 20 points as noise. The main grid needs more pairs and a test seed.